In [1]:
import uproot
import pandas as pd
import numpy as np
import glob

import matplotlib.pyplot as plt
plt.style.use('default')
plt.style.use('belle2')

import os

In [2]:
gen_MC_name = "/share/storage/jykim/storage_b2/storage/reduced_ntuples/MC15rd/etapip_eteeta/MC15rd_etaetapip_loose_v7_241213_temp"
base_path_sig = "/share/storage/jykim/storage_ghi/Ntuples_ghi_2/MC15rd_sigMC"
sig_MC_name = "241213_loose_v7"
overall_version = "v7_xgboost_angle_cut"
base_path_sig = "/share/storage/jykim/storage_ghi/Ntuples_ghi_2/MC15rd_sigMC"

import glob

# base_path = "/share/storage/jykim/storage_b2/storage/reduced_ntuples/MC15rd/etapip_eteeta/MC15rd_etaetapip_loose_v3_241129"
# base_path = "/share/storage/jykim/storage_b2/storage/reduced_ntuples/MC15rd/etapip_eteeta/MC15rd_etaetapip_loose_v6_241211"
# base_path = "/share/storage/jykim/storage_b2/storage/reduced_ntuples/MC15rd/etapip_eteeta/MC15rd_etaetapip_loose_v4_241211"
base_path = gen_MC_name


cm_elements = ["15rd_eta_e7_18_4S_v3", "15rd_eta_e20_b26_v1", "15rd_eta_e20_e26_4S_v2", "15rd_eta_e21_5S_scan_v1", "15rd_eta_mori_off_v1"]

file_list = []
for element in cm_elements:
    pattern = f"{base_path}/{element}/*.root"
    file_list += glob.glob(pattern)

# Initialize an empty list to hold DataFrames
dataframes = []
branches_all = ["__experiment__", "__run__", "__event__",\
             'Dp_M','Dp_isSignal','Dp_CMS_p','Dp_cosAngleBetweenMomentumAndVertexVector','Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane',\
             'Dp_acos_cosAngleBetweenMomentumAndVertexVector','Dp_acos_cosAngleBetweenMomentumAndVertexVectorInXYPlane',\
             'etapip_Eta_M','etapip_Eta_isSignal','etapip_Eta_daughterDiffOfPhi_0_1','etapip_Eta_daughterAngle_0_1','etapip_Eta_Easym','etapip_Eta_p',\
             'etapip_Eta_genMotherPDG','etapip_Eta_genMotherID','etapip_gamma1_p','etapip_gamma2_p','etapip_gamma1_clusterNHits','etapip_gamma2_clusterNHits',\
             'etapip_gamma1_clusterReg', 'etapip_gamma2_clusterReg',\
             'Pip_pionID','Pip_pionIDNN','Pip_mcPDG','Pip_dr','Pip_dr','Pip_p',\
             'Pip_genMotherPDG','Pip_genMotherID',\
             'ROE_Mgg','dM_pi0','ROE_Mgg_50MeV','dM_pi0_50MeV','ROE_Mgg_75MeV','dM_pi0_75MeV','ROE_Mgg_mask','dM_pi0_mask',\
             'ROE_Mgg_2','dM_pi0_2','ROE_Mgg_50MeV_2','dM_pi0_50MeV_2','ROE_Mgg_75MeV_2','dM_pi0_75MeV_2','ROE_Mgg_mask_2','dM_pi0_mask_2',\
             'veto_isSignal','veto_isSignal_50MeV','veto_isSignal_75MeV','veto_isSignal_mask',\
             'num_Dstar','num_Dstar_no_nan','CFT_qr','CFT_prob']
# Process each file
for file_name in file_list:
    # Load the ROOT file and tree
    file = uproot.open(file_name)
    tree = file["etapip_gg"]

    # Specify the branches you want to extract
    branches = ['Dp_dz','Pip_binaryP','Pip_pionIDNN','Pip_pionID','Dp_Psum','dM_pi0_mask_nonan','dM_pi0_mask_2_nonan','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_cosHelicityAngleMomentum','Pip_dr',"Dp_isSignal","Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane", "Pip_p","Dp_CMS_p","etapip_Eta_Easym","etapip_Eta_p","CFT_prob","Dp_M","num_Dstar_no_nan","etapip_Eta_daughterDiffOfPhi_0_1","etapip_Eta_daughterAngle_0_1"]  # Replace with actual branch names

    # Convert the selected branches to a Pandas DataFrame
    df_temp = tree.arrays(branches, library="pd")
    # df_temp = tree.arrays(library="pd")

    # Append the DataFrame to the list
    dataframes.append(df_temp)

df_bkg = pd.concat(dataframes, ignore_index=True)

df_bkg = df_bkg.query('Dp_isSignal!=1')
df_bkg = df_bkg.query('(Pip_genMotherID!=etapip_Eta_genMotherID) |  (Pip_genMotherPDG!=431  & Pip_genMotherPDG!=-431) | (etapip_Eta_genMotherPDG!=431 & etapip_Eta_genMotherPDG!=-431)')
df_bkg = df_bkg.query('Pip_p>0.4')
df_bkg = df_bkg.query('Dp_M> 1.6 & Dp_M<2.1')

df_bkg = df_bkg.query('abs(etapip_Eta_daughterDiffOfPhi_0_1)<1.8')
df_bkg = df_bkg.query('etapip_Eta_daughterAngle_0_1<1.6')

nan_columns = df_bkg.isnull().any()
print(nan_columns)

df_bkg.describe()

Dp_dz                                                 False
Pip_binaryP                                           False
Pip_pionIDNN                                          False
Pip_pionID                                            False
Dp_Psum                                               False
dM_pi0_mask_nonan                                     False
dM_pi0_mask_2_nonan                                   False
Pip_genMotherID                                       False
etapip_Eta_genMotherID                                False
Pip_genMotherPDG                                      False
etapip_Eta_genMotherPDG                               False
Dp_cosHelicityAngleMomentum                           False
Pip_dr                                                False
Dp_isSignal                                            True
Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane    False
Pip_p                                                 False
Dp_CMS_p                                

,Dp_dz,Pip_binaryP,Pip_pionIDNN,Pip_pionID,Dp_Psum,dM_pi0_mask_nonan,dM_pi0_mask_2_nonan,Pip_genMotherID,etapip_Eta_genMotherID,Pip_genMotherPDG,...,Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane,Pip_p,Dp_CMS_p,etapip_Eta_Easym,etapip_Eta_p,CFT_prob,Dp_M,num_Dstar_no_nan,etapip_Eta_daughterDiffOfPhi_0_1,etapip_Eta_daughterAngle_0_1
count,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,...,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07,1.556534e+07
mean,2.895576e-03,9.564636e-01,9.527288e-01,7.450784e-01,3.850318e+00,1.397437e+02,1.402172e+02,6.467284e+00,1.901735e+00,5.980690e+03,...,5.719710e-02,1.493748e+00,3.225473e+00,5.495279e-01,2.356571e+00,5.083926e-01,1.816518e+00,-9.837840e-01,-1.178410e-02,8.310675e-01
std,1.398400e-01,1.617272e-01,9.525533e-02,3.675672e-01,9.898755e-01,1.172979e+03,1.174906e+03,7.041774e+00,3.862217e+00,2.321877e+05,...,9.921813e-01,1.065729e+00,7.779410e-01,2.932073e-01,1.294770e+00,2.917536e-01,1.424552e-01,1.831348e-01,8.841252e-01,3.724589e-01
min,-2.799784e+01,2.598619e-275,6.000001e-01,0.000000e+00,2.167923e+00,1.100000e-02,1.100000e-02,0.000000e+00,0.000000e+00,-9.000211e+06,...,-1.000000e+00,4.000000e-01,2.500000e+00,6.426477e-08,5.321031e-01,4.605055e-04,1.600000e+00,-1.000000e+00,-1.799999e+00,6.807023e-02
25%,-2.020726e-02,9.999097e-01,9.665705e-01,4.994094e-01,3.205321e+00,3.397619e-02,3.496010e-02,2.000000e+00,0.000000e+00,2.300000e+01,...,-9.999747e-01,5.875969e-01,2.705748e+00,2.928246e-01,1.262686e+00,2.512469e-01,1.692514e+00,-1.000000e+00,-6.781644e-01,5.067760e-01
50%,1.831648e-03,1.000000e+00,9.989273e-01,9.851183e-01,3.683609e+00,9.742753e-02,1.009636e-01,4.000000e+00,0.000000e+00,2.300000e+01,...,9.909180e-01,1.122848e+00,2.999291e+00,5.834228e-01,2.280678e+00,5.122088e-01,1.796454e+00,-1.000000e+00,-2.293982e-02,7.772649e-01
75%,2.431838e-02,1.000000e+00,1.000000e+00,9.997777e-01,4.224454e+00,2.807736e-01,2.915520e-01,8.000000e+00,2.000000e+00,1.130000e+02,...,9.999904e-01,2.251219e+00,3.497718e+00,8.300870e-01,3.119837e+00,7.684181e-01,1.934025e+00,-1.000000e+00,6.543318e-01,1.119989e+00
max,3.203020e+01,1.000000e+00,1.000000e+00,1.000000e+00,9.660061e+01,1.000000e+04,1.000000e+04,1.170000e+02,7.200000e+01,9.030221e+06,...,1.000000e+00,8.968652e+01,7.551107e+01,9.893696e-01,2.087782e+01,9.995414e-01,2.100000e+00,6.000000e+00,1.800000e+00,1.600000e+00


In [3]:
elements_sig = ["Dptoetapip_gg", "Dptoetapip_gg_cc"]
project_name = sig_MC_name


file_list_sig = []
for element in elements_sig:
    pattern = f"{base_path_sig}/{element}/{project_name}/*.root"
    file_list_sig += glob.glob(pattern)

dataframes_signal = []

for file_name in file_list_sig:
    # Load the ROOT file and tree
    file = uproot.open(file_name)
    tree = file["etapip_gg"]

    # Specify the branches you want to extract
    #branches = ['Dp_Psum','dM_pi0_mask_nonan','dM_pi0_mask_2_nonan','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_cosHelicityAngleMomentum','Pip_dr',"Dp_isSignal","Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane", "Pip_p","Dp_CMS_p","etapip_Eta_Easym","etapip_Eta_p","CFT_prob","Dp_M","num_Dstar_no_nan","etapip_Eta_daughterDiffOfPhi_0_1","etapip_Eta_daughterAngle_0_1"]  # Replace with actual branch names
    # branches = ['Dp_Psum','dM_pi0_mask','dM_pi0_mask_2','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_cosHelicityAngleMomentum','Pip_dr',"Dp_isSignal","Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane", "Pip_p","Dp_CMS_p","etapip_Eta_Easym","etapip_Eta_p","CFT_prob","Dp_M","num_Dstar_no_nan","etapip_Eta_daughterDiffOfPhi_0_1","etapip_Eta_daughterAngle_0_1"]  # Replace with actual branch names
    branches = ['Dp_dz','Pip_pionIDNN','Pip_pionID','Dp_Psum','dM_pi0_mask_nonan','dM_pi0_mask_2_nonan','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_cosHelicityAngleMomentum','Pip_dr',"Dp_isSignal","Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane", "Pip_p","Dp_CMS_p","etapip_Eta_Easym","etapip_Eta_p","CFT_prob","Dp_M","num_Dstar_no_nan","etapip_Eta_daughterDiffOfPhi_0_1","etapip_Eta_daughterAngle_0_1"]  # Replace with actual branch names

    # Convert the selected branches to a Pandas DataFrame
    df_temp = tree.arrays(branches, library="pd")
    # df_temp = tree.arrays(library="pd")

    # Append the DataFrame to the list
    dataframes_signal.append(df_temp)


df_signal = pd.concat(dataframes_signal, ignore_index=True)
df_signal = df_signal.query('Dp_isSignal==1')

In [4]:
elements_sig = ["Dsptoetapip_gg", "Dsptoetapip_gg_cc"]
project_name = sig_MC_name

file_list_sig = []
for element in elements_sig:
    pattern = f"{base_path_sig}/{element}/{project_name}/*.root"
    file_list_sig += glob.glob(pattern)

dataframes_signal = []

for file_name in file_list_sig:
    # Load the ROOT file and tree
    file = uproot.open(file_name)
    tree = file["etapip_gg"]

    # Specify the branches you want to extract
    #branches = ['Dp_Psum','dM_pi0_mask_nonan','dM_pi0_mask_2_nonan','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_cosHelicityAngleMomentum','Pip_dr',"Dp_isSignal","Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane", "Pip_p","Dp_CMS_p","etapip_Eta_Easym","etapip_Eta_p","CFT_prob","Dp_M","num_Dstar_no_nan","etapip_Eta_daughterDiffOfPhi_0_1","etapip_Eta_daughterAngle_0_1"]  # Replace with actual branch names
    # branches = ['Dp_Psum','dM_pi0_mask','dM_pi0_mask_2','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_cosHelicityAngleMomentum','Pip_dr',"Dp_isSignal","Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane", "Pip_p","Dp_CMS_p","etapip_Eta_Easym","etapip_Eta_p","CFT_prob","Dp_M","num_Dstar_no_nan","etapip_Eta_daughterDiffOfPhi_0_1","etapip_Eta_daughterAngle_0_1"]  # Replace with actual branch names
    branches = ['Dp_dz','Pip_pionIDNN','Pip_pionID','Dp_Psum','dM_pi0_mask_nonan','dM_pi0_mask_2_nonan','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_cosHelicityAngleMomentum','Pip_dr',"Dp_isSignal","Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane", "Pip_p","Dp_CMS_p","etapip_Eta_Easym","etapip_Eta_p","CFT_prob","Dp_M","num_Dstar_no_nan","etapip_Eta_daughterDiffOfPhi_0_1","etapip_Eta_daughterAngle_0_1"]  # Replace with actual branch names

    # Convert the selected branches to a Pandas DataFrame
    df_temp = tree.arrays(branches, library="pd")
    # df_temp = tree.arrays(library="pd")

    # Append the DataFrame to the list
    dataframes_signal.append(df_temp)


df_signal_Dsp = pd.concat(dataframes_signal, ignore_index=True)
df_signal_Dsp = df_signal_Dsp.query('(Pip_genMotherID==etapip_Eta_genMotherID) & (Pip_genMotherPDG==431  | Pip_genMotherPDG==-431) & (etapip_Eta_genMotherPDG==431 | etapip_Eta_genMotherPDG==-431)')

unique_values = df_signal_Dsp['etapip_Eta_genMotherPDG'].unique()
print(unique_values)

df_signal = pd.concat([df_signal, df_signal_Dsp], ignore_index=True)
df_signal = df_signal.query('Pip_p>0.4 ')

df_signal = df_signal.query('abs(etapip_Eta_daughterDiffOfPhi_0_1)<1.8')
df_signal = df_signal.query('etapip_Eta_daughterAngle_0_1<1.6')


nan_columns = df_signal.isnull().any()
print(nan_columns)

[ 431. -431.]
Dp_dz                                                 False
Pip_pionIDNN                                          False
Pip_pionID                                            False
Dp_Psum                                               False
dM_pi0_mask_nonan                                     False
dM_pi0_mask_2_nonan                                   False
Pip_genMotherID                                       False
etapip_Eta_genMotherID                                False
Pip_genMotherPDG                                      False
etapip_Eta_genMotherPDG                               False
Dp_cosHelicityAngleMomentum                           False
Pip_dr                                                False
Dp_isSignal                                           False
Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane    False
Pip_p                                                 False
Dp_CMS_p                                              False
etapip_Eta_Easym          

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Ensure the signal and background datasets are balanced
min_size = min(len(df_signal), len(df_bkg))

# Sample an equal number of rows from both signal and background
df_signal_balanced = df_signal.sample(n=min_size, random_state=42)
df_bkg_balanced = df_bkg.sample(n=min_size, random_state=42)

# Assign labels: signal=1, background=0
df_signal_balanced['label'] = 1
df_bkg_balanced['label'] = 0

# Concatenate the signal and background datasets
df_combined = pd.concat([df_signal_balanced, df_bkg_balanced], ignore_index=True)

# Shuffle the combined dataset
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# Drop the specified columns
columns_to_drop = ['etapip_Eta_daughterDiffOfPhi_0_1','Pip_binaryP','Pip_pionIDNN','Dp_M','Pip_pionID','dM_pi0_mask_nonan','dM_pi0_mask_2_nonan','Pip_genMotherID','etapip_Eta_genMotherID','Pip_genMotherPDG','etapip_Eta_genMotherPDG','Dp_isSignal','Dp_CMS_p','CFT_prob', 'Dp_M', 'Pip_p', 'etapip_Eta_p', 'num_Dstar_no_nan','etapip_Eta_daughterAngle_0_1']
df_combined = df_combined.drop(columns=columns_to_drop)
df_combined.describe()

,Dp_dz,Dp_Psum,Dp_cosHelicityAngleMomentum,Pip_dr,Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane,etapip_Eta_Easym,label
count,3.616710e+06,3.616710e+06,3.616710e+06,3.616710e+06,3.616710e+06,3.616710e+06,3616710.0
mean,9.659058e-03,3.950867e+00,-1.632315e-01,1.266830e-02,4.424547e-01,4.794646e-01,0.5
std,1.033641e-01,8.738307e-01,5.548164e-01,3.349492e-02,8.923330e-01,2.816378e-01,0.5
min,-2.175213e+01,2.221546e+00,-9.999700e-01,1.841810e-09,-1.000000e+00,2.417258e-07,0.0
25%,-1.730619e-02,3.353439e+00,-6.881386e-01,1.579444e-03,-9.968897e-01,2.335552e-01,0.0
50%,6.363350e-03,3.835854e+00,-2.561598e-01,4.133194e-03,9.999826e-01,4.738693e-01,0.5
75%,3.180877e-02,4.396235e+00,3.346474e-01,1.208764e-02,9.999997e-01,7.290461e-01,1.0
max,1.755148e+01,8.214689e+01,9.999940e-01,9.999451e-01,1.000000e+00,9.805742e-01,1.0


In [9]:
plt.rcParams['font.family'] = 'DejaVu Sans'

from sklearn.model_selection import train_test_split

# Separate features and labels
X = df_combined.drop(columns='label')
y = df_combined['label']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
X_train

,Dp_dz,Dp_Psum,Dp_cosHelicityAngleMomentum,Pip_dr,Dp_cosAngleBetweenMomentumAndVertexVectorInXYPlane,etapip_Eta_Easym
2926419,0.030402,3.531033,-0.795729,0.000898,0.934023,0.136741
3257092,-0.064639,3.589599,0.464317,0.000591,0.999999,0.662818
1260630,0.018932,2.890807,-0.650469,0.001080,-1.000000,0.018795
1270300,0.036397,3.181816,-0.856274,0.015427,0.999999,0.690601
2902834,-0.034308,3.578366,-0.732263,0.056434,0.999999,0.570264
...,...,...,...,...,...,...
2356330,-0.018931,2.872573,-0.758461,0.004102,1.000000,0.336447
3511566,0.078920,4.850796,-0.785066,0.049925,0.999999,0.147976
2229084,0.057824,4.547361,0.097801,0.020672,1.000000,0.247833
2768307,-0.006636,4.967609,-0.227538,0.007599,0.999999,0.299902


In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

X = X_train
y = y_train

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# # Define the parameter grid for XGBoost
# param_grid = {
#     'n_estimators': [50, 100, 150],
#     'max_depth': [3, 5, 7],
#     'learning_rate': [0.01, 0.1, 0.2],
#     'subsample': [0.8, 1.0],
#     'colsample_bytree': [0.8, 1.0]
# }
param_grid = {
    'n_estimators': [300, 500, 700, 900, 1100],
    'max_depth': [5, 7, 9, 11],
    'learning_rate': [0.005, 0.01, 0.1],
    #'subsample': [0.8,0.9, 1.0],
    #'colsample_bytree': [0.8,0.9, 1.0]
}
# Initialize GridSearchCV with cross-validation
grid_search = GridSearchCV(
     estimator=xgb_model,
     param_grid=param_grid,
     scoring='accuracy',
     cv=5,  # 5-fold cross-validation
     verbose=1,
     n_jobs=24
)

# Perform the grid search
# grid_search.fit(X, y)



# # Get the best model and its parameters
# best_xgb_model = grid_search.best_estimator_
# print("Best Parameters:", grid_search.best_params_)
# print("Best Cross-Validation Accuracy:", grid_search.best_score_)
# # # Save the best model if needed
# import joblib
# joblib.dump(best_xgb_model, f'MC15rd_best_xgb_model_etapip_gg_loose_{overall_version}.pkl')


xgb_model.fit(X,y)
best_xgb_model = xgb_model
#joblib.dump(best_xgb_model, f'MC15rd_best_xgb_model_etapip_gg_loose_{overall_version}.pkl')